# Phase 3: Association Rule Mining (Apriori)

Tujuan:
- Menemukan frequent itemsets dan aturan asosiasi.
- Menghitung support, confidence, dan lift.
- Menyediakan tabel aturan terurut dan ringkasan singkat.

Input: ../data/processed_dataset.csv
Output:
- ../reports/3-association-rules.csv
- ../reports/3-association-rules.txt

In [9]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

SAMPLE_SIZE = None  # None = pakai semua data; isi angka untuk sampling
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print("Menggunakan sampel:", len(df))
else:
    print("Menggunakan semua data.")

print("Jumlah baris:", len(df))
print("Jumlah kolom:", df.shape[1])
display(df.head())

Menggunakan semua data.
Jumlah baris: 99994
Jumlah kolom: 80


,derived_msa_md,state_code,county_code,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,preapproval,lien_status,reverse_mortgage,loan_amount,...,loan_purpose_32,loan_purpose_4,loan_purpose_5,income_bracket_High,income_bracket_Low,income_bracket_Medium,income_bracket_Very High,loan_size_Large,loan_size_Medium,loan_size_Small
0,12580,md,24005.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,465000,...,0,0,0,0,0,0,1,1,0,0
1,22220,ar,5007.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,125000,...,0,0,0,0,0,0,1,0,0,1
2,43900,sc,45083.0,c,conventional:subordinate lien,single family (1-4 units):site-built,2,2,2,55000,...,0,1,0,0,0,0,0,0,0,1
3,19430,oh,39057.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,65000,...,0,0,0,0,1,0,0,0,0,1
4,41700,tx,48029.0,c,conventional:first lien,single family (1-4 units):site-built,2,1,2,125000,...,0,0,0,0,0,0,0,0,0,1


In [10]:
def cap_categories(series, top_n=10, other_label="other"):
    value_counts = series.value_counts(dropna=True)
    top = value_counts.nlargest(top_n).index
    return series.where(series.isin(top), other_label)

binary_prefixes = [
    "derived_sex_",
    "derived_race_",
    "derived_ethnicity_",
    "loan_type_",
    "loan_purpose_",
    "purchaser_type_",
    "income_bracket_",
    "loan_size_",
]

binary_cols = [c for c in df.columns if any(c.startswith(p) for p in binary_prefixes)]
binary_df = df[binary_cols].copy()

# Pastikan format 0/1
binary_df = binary_df.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
binary_df = binary_df.clip(0, 1)

cat_cols = [
    "state_code",
    "conforming_loan_limit",
    "derived_loan_product_type",
    "derived_dwelling_category",
    "preapproval",
    "lien_status",
    "reverse_mortgage",
    "occupancy_type",
    "construction_method",
    "total_units",
    "applicant_age",
    "co_applicant_age",
    "applicant_age_above_62",
    "debt_to_income_ratio",
]
cat_cols = [c for c in cat_cols if c in df.columns]
cat_df = df[cat_cols].copy()

high_card_cols = {"state_code", "derived_loan_product_type", "derived_dwelling_category"}
sentinel_values = {"8888", "9999", "8888.0", "9999.0"}
sentinel_cols = ["applicant_age", "co_applicant_age"]
for col in cat_df.columns:
    cat_df[col] = cat_df[col].astype(str).str.strip().str.lower()
    cat_df[col] = cat_df[col].replace({"nan": "missing", "": "missing"})
    if col in sentinel_cols:
        cat_df[col] = cat_df[col].replace(sentinel_values, "missing")
    if col in high_card_cols:
        cat_df[col] = cap_categories(cat_df[col], top_n=10, other_label="other")

cat_ohe = pd.get_dummies(cat_df, prefix=cat_df.columns, dtype=int)

num_bin_cols = {
    "loan_amount": 4,
    "income": 4,
    "interest_rate": 4,
    "combined_loan_to_value_ratio": 4,
    "loan_term": 3,
}
bin_df = pd.DataFrame(index=df.index)
for col, q in num_bin_cols.items():
    if col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")
        if series.dropna().nunique() < 2:
            print(f"Lewati binning '{col}': variasi terlalu sedikit.")
            continue
        try:
            bin_df[f"{col}_bin"] = pd.qcut(series, q=q, duplicates="drop")
        except ValueError as exc:
            print(f"Lewati binning '{col}': {exc}")

if not bin_df.empty:
    bin_df = bin_df.astype("string").fillna("missing").astype(str)
    bin_ohe = pd.get_dummies(bin_df, prefix=bin_df.columns, dtype=int)
else:
    bin_ohe = pd.DataFrame(index=df.index)

item_df = pd.concat([binary_df, cat_ohe, bin_ohe], axis=1)
item_df = item_df.loc[:, item_df.sum(axis=0) > 0]

redundant_prefixes = ("income_bin_", "loan_amount_bin_")
redundant_cols = [c for c in item_df.columns if c.startswith(redundant_prefixes)]
if len(redundant_cols) > 0:
    item_df = item_df.drop(columns=redundant_cols)
    print("Kolom redundan dihapus:", len(redundant_cols))

missing_cols = item_df.columns[item_df.columns.str.endswith("_missing")]
if len(missing_cols) > 0:
    item_df = item_df.drop(columns=missing_cols)
    print("Kolom missing dihapus:", len(missing_cols))

if item_df.empty:
    raise ValueError("Itemset kosong. Cek kolom input atau kurangi filter.")

print("Jumlah item awal:", item_df.shape[1])

Kolom redundan dihapus: 9
Kolom missing dihapus: 8
Jumlah item awal: 135


In [11]:
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError as exc:
    raise ImportError("Paket mlxtend belum terpasang. Instal dulu: pip install mlxtend") from exc

MIN_SUPPORT = 0.02
MIN_CONF = 0.6
MIN_LIFT = 1.5
MAX_ITEMSETS = 150
MAX_LEN = 3

CORRELATED_GROUPS = [
    {"loan_type_", "derived_loan_product_type_"},
    {"lien_status_", "derived_loan_product_type_"},
    {"applicant_age_", "applicant_age_above_62_"},
    {"co_applicant_age_", "applicant_age_"},
    {"conforming_loan_limit_", "loan_size_"},
    {"preapproval_", "loan_purpose_"},
]

def is_tautological(row):
    items = set(row["antecedents"]).union(row["consequents"])
    for group in CORRELATED_GROUPS:
        prefixes_found = [p for p in group if any(str(item).startswith(p) for item in items)]
        if len(prefixes_found) >= 2:
            return True
    return False

def apply_filters(rules_df, min_lift):
    rules_df = rules_df[rules_df["lift"] >= min_lift]
    rules_df = rules_df[rules_df["consequents"].apply(lambda x: len(x) == 1)]
    tautology_mask = rules_df.apply(is_tautological, axis=1)
    tautology_count = int(tautology_mask.sum())
    rules_df = rules_df[~tautology_mask]
    return rules_df, tautology_count

# Drop item dengan support sangat rendah agar proses lebih ringan
item_support = item_df.mean(axis=0)
item_df_filtered = item_df.loc[:, item_support >= MIN_SUPPORT]

if item_df_filtered.shape[1] == 0:
    raise ValueError("Tidak ada item dengan support >= MIN_SUPPORT. Turunkan MIN_SUPPORT.")

# Batasi jumlah item agar Apriori tidak memakan memori berlebihan
if item_df_filtered.shape[1] > MAX_ITEMSETS:
    top_items = item_support.sort_values(ascending=False).head(MAX_ITEMSETS).index
    item_df_filtered = item_df_filtered[top_items]
    print("Item dibatasi ke:", len(top_items))

# Ubah ke bool agar lebih hemat memori
item_df_filtered = item_df_filtered.astype(bool)

print("Jumlah item setelah filter support:", item_df_filtered.shape[1])

frequent_itemsets = apriori(
    item_df_filtered,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_LEN,
    low_memory=True,
 )
print("Frequent itemsets:", len(frequent_itemsets))

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONF)
rules, tautology_removed = apply_filters(rules, MIN_LIFT)

if len(rules) < 10:
    relax_confs = [0.55, 0.5, 0.45]
    for conf in relax_confs:
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=conf)
        rules, tautology_removed = apply_filters(rules, MIN_LIFT)
        if len(rules) >= 10:
            MIN_CONF = conf
            break

print("Jumlah rules setelah filter:", len(rules))
print("Rules terhapus (tautologi):", tautology_removed)
print("Threshold akhir -> support:", MIN_SUPPORT, "confidence:", MIN_CONF, "lift:", MIN_LIFT)

Jumlah item setelah filter support: 97
Frequent itemsets: 17678
Jumlah rules setelah filter: 566
Rules terhapus (tautologi): 415
Threshold akhir -> support: 0.02 confidence: 0.6 lift: 1.5


In [12]:
def format_itemset(itemset):
    return ", ".join(sorted([str(x) for x in itemset]))

def parse_itemset_str(text):
    if not isinstance(text, str) or text.strip() == "":
        return []
    return [t.strip() for t in text.split(",")]

MINOR_PREFIXES = ("total_units_", "occupancy_type_", "derived_race_")

def canonical_item(item):
    text = str(item).strip().lower()
    if text == "lien_status_2":
        return "subordinate_lien"
    if "subordinate lien" in text:
        return "subordinate_lien"
    return text

def canonicalize_items(items):
    return [canonical_item(i) for i in items]

def core_antecedent_key(items):
    canon = canonicalize_items(items)
    core = [i for i in canon if not i.startswith(MINOR_PREFIXES)]
    return tuple(sorted(core))

def rule_signature(antecedent_items, consequent_items):
    ant_key = core_antecedent_key(antecedent_items)
    cons_key = tuple(sorted(canonicalize_items(consequent_items)))
    return (ant_key, cons_key)

CONSEQUENT_GROUPS = [
    {"name": "loan_product", "prefixes": {"loan_type_", "derived_loan_product_type_", "lien_status_"}},
    {"name": "income_group", "prefixes": {"income_bracket_", "income_bin_"}},
    {"name": "loan_size_group", "prefixes": {"loan_size_", "loan_amount_bin_", "conforming_loan_limit_"}},
    {"name": "preapproval_purpose", "prefixes": {"preapproval_", "loan_purpose_"}},
    {"name": "age_group", "prefixes": {"applicant_age_", "co_applicant_age_", "applicant_age_above_62_"}},
    {"name": "dwelling_construction", "prefixes": {"construction_method_", "derived_dwelling_category_"}},
]

REPORT_MIN_LIFT = 2.0

def get_consequent_key(consequent):
    text = str(consequent).strip().lower()
    for group in CONSEQUENT_GROUPS:
        if any(text.startswith(p) for p in group["prefixes"]):
            return group["name"]
    return f"cons_{text}"

def init_state():
    return {"selected": [], "used_signatures": set(), "used_ants": set(), "group_counts": {}}

def add_rules(base_df, caps, state, top_n, allow_over_cap=False):
    for _, row in base_df.iterrows():
        if len(state["selected"]) >= top_n:
            break
        ant_items = row.get("antecedent_items")
        cons_items = row.get("consequent_items")
        if not isinstance(ant_items, list):
            ant_items = parse_itemset_str(row.get("antecedents", ""))
        if not isinstance(cons_items, list):
            cons_items = parse_itemset_str(row.get("consequents", ""))
        ant_key = core_antecedent_key(ant_items)
        if ant_key in state["used_ants"]:
            continue
        signature = rule_signature(ant_items, cons_items)
        if signature in state["used_signatures"]:
            continue
        cons_key = get_consequent_key(cons_items[0] if cons_items else "")
        if caps is not None:
            cap = caps.get(cons_key, 1)
            if not allow_over_cap and state["group_counts"].get(cons_key, 0) >= cap:
                continue
        state["selected"].append(row)
        state["used_signatures"].add(signature)
        state["used_ants"].add(ant_key)
        state["group_counts"][cons_key] = state["group_counts"].get(cons_key, 0) + 1
    return state

def select_report_rules(df_rules, top_n=10):
    base = df_rules[df_rules["lift"] >= REPORT_MIN_LIFT].copy()
    if base.empty:
        return pd.DataFrame(), REPORT_MIN_LIFT
    base = base.sort_values(by=["lift", "confidence", "support"], ascending=False)
    state = init_state()
    strict_caps = {
        "dwelling_construction": 1,
        "loan_product": 2,
        "loan_size_group": 2,
        "income_group": 2,
    }
    state = add_rules(base, strict_caps, state, top_n)
    if len(state["selected"]) < top_n:
        relaxed_caps = dict(strict_caps)
        relaxed_caps["loan_product"] = 3
        relaxed_caps["loan_size_group"] = 3
        relaxed_caps["income_group"] = 3
        state = add_rules(base, relaxed_caps, state, top_n)
    if len(state["selected"]) < top_n:
        state = add_rules(base, {}, state, top_n, allow_over_cap=True)
    return pd.DataFrame(state["selected"]), REPORT_MIN_LIFT

def safe_write_csv(df, path):
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        alt_path = path.replace(".csv", "_new.csv")
        df.to_csv(alt_path, index=False)
        return alt_path

def safe_write_txt(lines, path):
    try:
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return path
    except PermissionError:
        alt_path = path.replace(".txt", "_new.txt")
        with open(alt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return alt_path

rules_table = rules.copy()
rules_table["antecedent_items"] = rules_table["antecedents"].apply(lambda x: sorted([str(i) for i in x]))
rules_table["consequent_items"] = rules_table["consequents"].apply(lambda x: sorted([str(i) for i in x]))
rules_table["antecedents"] = rules_table["antecedent_items"].apply(lambda items: ", ".join(items))
rules_table["consequents"] = rules_table["consequent_items"].apply(lambda items: ", ".join(items))

rules_table = rules_table[[
    "antecedents",
    "consequents",
    "support",
    "confidence",
    "lift",
    "antecedent_items",
    "consequent_items",
]]
rules_table = rules_table.sort_values(by=["lift", "confidence", "support"], ascending=False)

rules_for_selection = rules_table.copy()
rules_table = rules_table[["antecedents", "consequents", "support", "confidence", "lift"]]

top_rules, used_min_lift = select_report_rules(rules_for_selection, top_n=10)
top_rules_display = top_rules[["antecedents", "consequents", "support", "confidence", "lift"]] if not top_rules.empty else top_rules
print(f"Top 10 aturan untuk laporan (lift >= {used_min_lift}):")
display(top_rules_display)

rules_csv_path = "../reports/3-association-rules.csv"
rules_txt_path = "../reports/3-association-rules.txt"

csv_path_used = safe_write_csv(rules_table, rules_csv_path)

if len(top_rules_display) < 10:
    print("Peringatan: aturan untuk laporan kurang dari 10. Pertimbangkan menambah item atau melonggarkan filter.")

lines = []
lines.append("Ringkasan Aturan Asosiasi (Top 10)")
lines.append("")
for _, row in top_rules_display.iterrows():
    lines.append(
        f"- Jika {row['antecedents']} maka {row['consequents']} "
        f"(support={row['support']:.3f}, confidence={row['confidence']:.3f}, lift={row['lift']:.2f})"
    )

txt_path_used = safe_write_txt(lines, rules_txt_path)

print("Tabel rules disimpan ke:", csv_path_used)
print("Ringkasan disimpan ke:", txt_path_used)

Top 10 aturan untuk laporan (lift >= 2.0):


,antecedents,consequents,support,confidence,lift
22620,"conforming_loan_limit_c, construction_method_2",derived_dwelling_category_single family (1-4 u...,0.043773,1.000000,22.829680
16201,"combined_loan_to_value_ratio_bin_(60.656, 78.1...",derived_loan_product_type_conventional:subordi...,0.020601,0.888697,5.255138
16197,"applicant_age_35-44, loan_purpose_2",derived_loan_product_type_conventional:subordi...,0.020851,0.874581,5.171662
7441,"conforming_loan_limit_nc, derived_ethnicity_no...",income_bracket_Very High,0.024941,0.869292,3.939989
4970,"conforming_loan_limit_nc, derived_race_white",income_bracket_Very High,0.020051,0.867965,3.933974
17738,"income_bracket_Low, lien_status_2",loan_size_Small,0.026742,0.952279,2.702796
18144,derived_loan_product_type_conventional:subordi...,loan_size_Small,0.034032,0.938759,2.664421
16077,"income_bracket_Very High, loan_purpose_2",derived_loan_product_type_conventional:subordi...,0.023621,0.874491,5.171132
15216,"conforming_loan_limit_nc, loan_purpose_1",income_bracket_Very High,0.023021,0.828952,3.757150
20228,"applicant_age_25-34, lien_status_2",loan_size_Small,0.020021,0.907937,2.576941


Tabel rules disimpan ke: ../reports/3-association-rules.csv
Ringkasan disimpan ke: ../reports/3-association-rules.txt
